<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/lab-04-memory-for-the-lumina-assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 (graded) — Memory for the Lumina assistant
**Course 3: AI Agents and Agentic AI with Python — Chapter 4: Memory & context management**

**Problem brief (Leo Farkas, Lumina Health):** "A clinician shouldn't have to re-explain
their patient panel every session. But the assistant also can't hold a 200-page history in
the prompt."

**What you'll submit:** long-term vector memory, a running-summary compactor, a persisted
profile, cross-session recall demonstrated, and a cost/quality comparison against a naive
"keep everything" baseline with a token-budget breakdown.

In [ ]:
!pip install -q sentence-transformers chromadb tiktoken

## 1. A multi-session transcript

In [ ]:
sessions = [
    ['Clinician: My patient panel today includes three type-2 diabetics on metformin.',
     'Assistant: Noted. Anything specific about them I should remember for follow-up?',
     'Clinician: One of them, patient A, has reported occasional dizziness after doses.'],
    ['Clinician: Following up on patient A from last time — any general guidance on dizziness with metformin?',
     'Assistant: Dizziness can sometimes relate to timing relative to meals; worth reviewing.'],
    ['Clinician: New patient today, patient B, recently started on insulin.',
     'Assistant: Understood, noting patient B on insulin.'],
    ['Clinician: Remind me — what did we note about patient A again?'],  # cross-session recall test
]

## 2. Long-term vector memory (Chroma)

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()
memory_collection = chroma_client.get_or_create_collection('lumina_memory')

def write_memory(text, session_id, turn_id):
    emb = embedder.encode([text], normalize_embeddings=True)[0]
    memory_collection.add(
        ids=[f's{session_id}-t{turn_id}'], embeddings=[emb.tolist()],
        documents=[text], metadatas=[{'session_id': session_id}],
    )

def recall_memory(query, top_k=2):
    emb = embedder.encode([query], normalize_embeddings=True)[0]
    res = memory_collection.query(query_embeddings=[emb.tolist()], n_results=top_k)
    return res['documents'][0] if res['documents'] else []

## 3. A structured profile (facts worth remembering precisely, not just semantically)

In [ ]:
import re

patient_profile = {}

def update_profile(turn_text):
    """A simple write policy: extract 'patient X ...' mentions into a structured store —
    a real system would use an LLM call here; the pattern (extract durable facts, don't
    store everything verbatim) is what matters for this lab."""
    for m in re.finditer(r'patient ([A-Z])[,.]?\s*(.{0,80})', turn_text):
        pid, note = m.group(1), m.group(2).strip()
        patient_profile.setdefault(pid, []).append(note)

## 4. Running-summary compaction

In [ ]:
running_summary = ''

def compact_into_summary(turns, existing_summary):
    """A cheap extractive compactor for this lab (no LLM call needed): keep the summary to
    the most information-dense sentence per turn batch. A production system would use an
    LLM summarization call here instead — the compaction PATTERN is the point."""
    new_facts = [t.split(': ', 1)[-1] for t in turns if t.startswith('Clinician:')]
    combined = existing_summary + ' ' + ' '.join(new_facts)
    return combined.strip()[-500:]  # bounded — this is what keeps the summary from growing forever

## 5. Run the multi-session transcript through the full memory system

In [ ]:
for session_id, turns in enumerate(sessions):
    for turn_id, turn in enumerate(turns):
        write_memory(turn, session_id, turn_id)
        update_profile(turn)
    running_summary = compact_into_summary(turns, running_summary)

print('Structured profile:', patient_profile)
print('\nRunning summary:', running_summary)

## 6. Cross-session recall test

In [ ]:
query = 'What did we note about patient A again?'
recalled = recall_memory(query)
print('Recalled from vector memory:')
for r in recalled:
    print(' -', r)
print('\nFrom structured profile:', patient_profile.get('A'))

assert any('dizziness' in r.lower() for r in recalled) or any('dizz' in n.lower() for n in patient_profile.get('A', [])), \
    'Expected the dizziness note about patient A to be recalled from a prior session.'
print('\nCross-session recall PASSED.')

## 7. Cost/quality vs. a naive "keep everything" baseline

In [ ]:
import tiktoken

enc = tiktoken.get_encoding('cl100k_base')

def count_tokens(text):
    return len(enc.encode(text))

all_turns_text = '\n'.join(t for s in sessions for t in s)
naive_prompt = all_turns_text + '\n' + query

memory_prompt = (
    f'Profile: {patient_profile}\nSummary: {running_summary}\n'
    f'Relevant memories: {recalled}\nCurrent question: {query}'
)

naive_tokens = count_tokens(naive_prompt)
memory_tokens = count_tokens(memory_prompt)

print(f'Naive "keep everything" prompt: {naive_tokens} tokens')
print(f'Memory-managed prompt:          {memory_tokens} tokens')
print(f'Reduction: {(1 - memory_tokens/naive_tokens):.0%}')
print('\nToken budget breakdown for the memory-managed prompt:')
print(f'  profile:            {count_tokens(str(patient_profile))} tokens')
print(f'  running summary:    {count_tokens(running_summary)} tokens')
print(f'  retrieved memories: {count_tokens(str(recalled))} tokens')
print(f'  current question:   {count_tokens(query)} tokens')

## 8. Write-up (fill in)
At what transcript length would the naive baseline start hitting real problems (cost,
"lost in the middle" quality loss, or a hard context-window limit)? Would you trust the
extractive compactor above for a real clinical use case, or does that specific task need an
LLM-based summarizer instead — why?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 4: Memory & context management*